# 01. データ収集：国会会議録APIからDEI関連発言を取得

## 概要
国会会議録検索システム（国立国会図書館提供）のAPIを使用し、
DEIに関連する国会議員の発言を収集する。

## データソース
- **API**: 国会会議録検索システム
- **エンドポイント**: `https://kokkai.ndl.go.jp/api/speech`
- **認証**: 不要（オープンAPI）
- **制限**: 1回100件まで・1000件超はエラー → 期間分割で対応

## 取得方針
- **期間**: 2020年〜2025年
- **対象**: 衆議院・参議院・委員会
- **キーワード**: ダイバーシティ、多様性、女性活躍、包括、公平、ジェンダー、障害者、外国人労働者、DEI

## 出力
- `data/speeches_raw.csv`

In [3]:
# ライブラリのインポート
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import time
import os

# データ保存先ディレクトリの作成
os.makedirs("../data", exist_ok=True)

print("ライブラリのインポート完了")

ライブラリのインポート完了


## 1. API動作確認
まず1件だけ取得してAPIの動作とレスポンス構造を確認する。

In [4]:
# API動作確認：ダイバーシティで1件だけ取得してみる
url = "https://kokkai.ndl.go.jp/api/speech"

params = {
    "any": "ダイバーシティ",
    "from": "2024-01-01",
    "until": "2024-03-31",
    "maximumRecords": 1,
    "recordPacking": "json"
}

response = requests.get(url, params=params)
print(f"ステータスコード: {response.status_code}")
print(f"レスポンス内容:\n{response.text[:500]}")

ステータスコード: 200
レスポンス内容:
{"numberOfRecords":18,"numberOfReturn":1,"startRecord":1,"nextRecordPosition":2,"speechRecord":[{"speechID":"121315007X00220240321_115","issueID":"121315007X00220240321","imageKind":"会議録","searchObject":115,"session":213,"nameOfHouse":"参議院","nameOfMeeting":"農林水産委員会","issue":"第2号","date":"2024-03-21","closing":null,"speechOrder":115,"speaker":"松野明美","speakerYomi":"まつのあけみ","speakerGroup":"日本維新の会・教育無償化を実現する会","speakerPosition":null,"speakerRole":null,"speech":"○松野明美君　活発に活動なさっているなということは分かりましたが、知らない方


## 2. DEIキーワードの設定と大量取得
9つのキーワード × 2020〜2025年を半年ごとに分割して取得する。

In [4]:
# ============================================================
# DEIキーワードと取得期間の設定
# ============================================================

# DEI関連キーワード一覧
KEYWORDS = [
    "ダイバーシティ",
    "多様性",
    "女性活躍",
    "包括",
    "公平",
    "ジェンダー",
    "障害者雇用",
    "外国人労働者",
    "DEI"
]

# 取得期間（1000件制限を避けるため半年ごとに分割）
PERIODS = [
    ("2020-01-01", "2020-06-30"),
    ("2020-07-01", "2020-12-31"),
    ("2021-01-01", "2021-06-30"),
    ("2021-07-01", "2021-12-31"),
    ("2022-01-01", "2022-06-30"),
    ("2022-07-01", "2022-12-31"),
    ("2023-01-01", "2023-06-30"),
    ("2023-07-01", "2023-12-31"),
    ("2024-01-01", "2024-06-30"),
    ("2024-07-01", "2024-12-31"),
    ("2025-01-01", "2025-06-30"),
    ("2025-07-01", "2025-12-31"),
]

print(f"キーワード数: {len(KEYWORDS)}")
print(f"取得期間数: {len(PERIODS)}")
print(f"最大リクエスト数: {len(KEYWORDS) * len(PERIODS)}")

キーワード数: 9
取得期間数: 12
最大リクエスト数: 108


## 3. データ取得関数の定義
APIから発言データを取得してDataFrameに変換する関数を定義する。

In [5]:
# ============================================================
# API取得関数の定義
# ============================================================

def fetch_speeches(keyword, from_date, until_date, max_records=100):
    """
    国会会議録APIから発言データを取得する関数
    
    Parameters:
        keyword    : 検索キーワード
        from_date  : 取得開始日 (例: "2020-01-01")
        until_date : 取得終了日 (例: "2020-06-30")
        max_records: 1回のリクエストで取得する最大件数（上限100）
    
    Returns:
        list: 発言データのリスト
    """
    url = "https://kokkai.ndl.go.jp/api/speech"
    speeches = []
    start_record = 1  # 取得開始位置

    while True:
        params = {
            "any": keyword,
            "from": from_date,
            "until": until_date,
            "maximumRecords": max_records,
            "startRecord": start_record,
            "recordPacking": "json"
        }

        response = requests.get(url, params=params)
        
        # レスポンスが正常でない場合はスキップ
        if response.status_code != 200:
            print(f"エラー: {keyword} {from_date}〜{until_date} status={response.status_code}")
            break

        data = response.json()
        records = data.get("speechRecord", [])
        
        # 取得件数が0件なら終了
        if not records:
            break

        # 必要な項目だけ抽出して格納
        for r in records:
            speeches.append({
                "keyword"    : keyword,
                "date"       : r.get("date"),
                "nameOfHouse": r.get("nameOfHouse"),
                "nameOfMeeting": r.get("nameOfMeeting"),
                "speaker"    : r.get("speaker"),
                "speakerGroup": r.get("speakerGroup"),
                "speech"     : r.get("speech"),
            })

        # 次のページがなければ終了
        next_pos = data.get("nextRecordPosition")
        if not next_pos:
            break

        # 次のページへ
        start_record = next_pos
        time.sleep(0.5)  # サーバー負荷軽減のため0.5秒待機

    return speeches

print("fetch_speeches関数の定義完了")

fetch_speeches関数の定義完了


## 4. データ取得実行
全キーワード × 全期間でデータを取得する。
取得件数をリアルタイムで確認しながら進める。

In [ ]:
# ============================================================
# 全キーワード × 全期間でデータ取得
# ============================================================

all_speeches = []  # 全発言データを格納するリスト

for keyword in KEYWORDS:
    for from_date, until_date in PERIODS:
        speeches = fetch_speeches(keyword, from_date, until_date)
        all_speeches.extend(speeches)
        print(f"[取得完了] {keyword} | {from_date}〜{until_date} | {len(speeches)}件")
        time.sleep(0.5)  # サーバー負荷軽減

print(f"\n総取得件数: {len(all_speeches)}件")

In [6]:
# ============================================================
# 全キーワード × 全期間でデータ取得
# ============================================================

all_speeches = []  # 全発言データを格納するリスト

for keyword in KEYWORDS:
    for from_date, until_date in PERIODS:
        speeches = fetch_speeches(keyword, from_date, until_date)
        all_speeches.extend(speeches)
        print(f"[取得完了] {keyword} | {from_date}〜{until_date} | {len(speeches)}件")
        time.sleep(0.5)  # サーバー負荷軽減

print(f"\n総取得件数: {len(all_speeches)}件")

[取得完了] ダイバーシティ | 2020-01-01〜2020-06-30 | 15件
[取得完了] ダイバーシティ | 2020-07-01〜2020-12-31 | 3件
[取得完了] ダイバーシティ | 2021-01-01〜2021-06-30 | 39件
[取得完了] ダイバーシティ | 2021-07-01〜2021-12-31 | 0件
[取得完了] ダイバーシティ | 2022-01-01〜2022-06-30 | 30件
[取得完了] ダイバーシティ | 2022-07-01〜2022-12-31 | 3件
[取得完了] ダイバーシティ | 2023-01-01〜2023-06-30 | 25件
[取得完了] ダイバーシティ | 2023-07-01〜2023-12-31 | 11件
[取得完了] ダイバーシティ | 2024-01-01〜2024-06-30 | 30件
[取得完了] ダイバーシティ | 2024-07-01〜2024-12-31 | 2件
[取得完了] ダイバーシティ | 2025-01-01〜2025-06-30 | 46件
[取得完了] ダイバーシティ | 2025-07-01〜2025-12-31 | 0件
[取得完了] 多様性 | 2020-01-01〜2020-06-30 | 157件
[取得完了] 多様性 | 2020-07-01〜2020-12-31 | 92件
[取得完了] 多様性 | 2021-01-01〜2021-06-30 | 425件
[取得完了] 多様性 | 2021-07-01〜2021-12-31 | 27件
[取得完了] 多様性 | 2022-01-01〜2022-06-30 | 366件
[取得完了] 多様性 | 2022-07-01〜2022-12-31 | 123件
[取得完了] 多様性 | 2023-01-01〜2023-06-30 | 617件
[取得完了] 多様性 | 2023-07-01〜2023-12-31 | 94件
[取得完了] 多様性 | 2024-01-01〜2024-06-30 | 665件
[取得完了] 多様性 | 2024-07-01〜2024-12-31 | 67件
[取得完了] 多様性 | 2025-01-01〜2025-06-30 | 329件
[取得完了] 

## 5. DataFrameに変換・重複除去・CSV保存

### 重複除去の方針
同一発言テキスト（`speech`）が複数のキーワードでヒットする場合がある。
例：「ダイバーシティ推進と女性活躍について」という発言は
「ダイバーシティ」「女性活躍」の両キーワードでヒットする。

クラスタリングは発言内容で分類するため、同じ発言が複数存在すると：
- 多数派がさらに強調される
- 少数派が相対的に埋もれる

→ **同一発言をグループ化し、該当キーワードを全て結合して1行にまとめる**
→ 例：keyword = "ダイバーシティ|女性活躍"
→ **全発言を公平に1票として扱う**

In [8]:
# ============================================================
# DataFrameに変換
# ============================================================
df = pd.DataFrame(all_speeches)
print(f"変換後のデータ数: {len(df)}件")

# ============================================================
# 重複除去
# 同一発言テキストをグループ化しキーワードを全て結合して1行にまとめる
# 例：keyword = "ダイバーシティ|女性活躍"
# ============================================================
df = df.groupby("speech", as_index=False).agg({
    "keyword"       : lambda x: "|".join(sorted(set(x))),
    "date"          : "first",
    "nameOfHouse"   : "first",
    "nameOfMeeting" : "first",
    "speaker"       : "first",
    "speakerGroup"  : "first",
})
print(f"重複除去後のデータ数: {len(df)}件")

# ============================================================
# CSV保存
# ============================================================
output_path = "../data/speeches_raw.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"保存完了: {output_path}")
print(df.head(3))

変換後のデータ数: 19388件
重複除去後のデータ数: 18259件
保存完了: ../data/speeches_raw.csv
                                              speech keyword        date  \
0  ○あかま国務大臣　おはようございます。\r\n　国家公安委員会委員長として、拉致問題に関する...      包括  2025-12-02   
1  ○あかま国務大臣　吉川委員にお答えいたします。\r\n　被災者生活再建支援金というのは、災害...      公平  2025-12-04   
2  ○あかま委員長　これより会議を開きます。\r\n　国土交通行政の基本施策に関する件について調...   障害者雇用  2021-05-26   

  nameOfHouse         nameOfMeeting speaker speakerGroup  
0         衆議院  北朝鮮による拉致問題等に関する特別委員会   あかま二郎  自由民主党・無所属の会  
1         衆議院             災害対策特別委員会   あかま二郎  自由民主党・無所属の会  
2         衆議院               国土交通委員会   あかま二郎  自由民主党・無所属の会  


In [9]:
# ============================================================
# データの基本確認
# ============================================================

print("=== 基本情報 ===")
print(f"総発言数: {len(df)}件")
print(f"期間: {df['date'].min()} 〜 {df['date'].max()}")
print(f"\n欠損値:\n{df.isnull().sum()}")

print("\n=== キーワード別件数 ===")
print(df['keyword'].value_counts())

print("\n=== 院別件数 ===")
print(df['nameOfHouse'].value_counts())

print("\n=== 会派別件数（上位10） ===")
print(df['speakerGroup'].value_counts().head(10))

=== 基本情報 ===
総発言数: 18259件
期間: 2020-01-20 〜 2025-12-18

欠損値:
speech              0
keyword             0
date                0
nameOfHouse         0
nameOfMeeting       0
speaker             0
speakerGroup     3965
dtype: int64

=== キーワード別件数 ===
keyword
公平                      7512
包括                      4054
多様性                     2522
ジェンダー                    982
女性活躍                     883
                        ... 
包括|外国人労働者|多様性              1
ジェンダー|外国人労働者|多様性           1
DEI|ジェンダー|ダイバーシティ|公平       1
DEI|女性活躍                   1
ダイバーシティ|多様性|障害者雇用          1
Name: count, Length: 77, dtype: int64

=== 院別件数 ===
nameOfHouse
衆議院    9644
参議院    8605
両院       10
Name: count, dtype: int64

=== 会派別件数（上位10） ===
speakerGroup
自由民主党・無所属の会     4002
自由民主党           1696
立憲民主党・無所属       1687
日本共産党           1048
公明党             1043
日本維新の会           949
立憲民主・社民          863
自由民主党・国民の声       446
国民民主党・無所属クラブ     424
国民民主党・新緑風会       390
Name: count, dtype: int64


## 6. データ基本確認の結果と考察

### 観察結果
- **総発言数**: 18,259件
- **期間**: 2020-01-20 〜 2025-12-18
- **欠損値**: `speakerGroup`に3,965件の欠損 → 参考人・大臣等の無所属発言と推測

### キーワード偏りの考察
「公平」（7,512件）が突出して多い。
→ 「公平」は日常的な文脈でも使われる汎用語のため、DEI文脈以外の発言が混入している可能性がある
→ 前処理段階でDEI文脈に絞り込む必要があるかもしれない

### 次のステップ
- `speakerGroup`の欠損値の扱いを決める
- 「公平」の発言内容を実際に確認する
- 02_eda.ipynbで詳細分析を行う